# Telco Customer Churn Prediction
**ACM SIG AI Recruitment Task**  
End-to-End Machine Learning Pipeline: EDA, Preprocessing, Stratified Splitting, Logistic Regression, Threshold Tuning, and Subgroup Error Analysis.

## Phase 1 & 2: Environment Setup, Dataset Loading & Verification

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set matplotlib inline style
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# Load raw dataset
raw_csv = os.path.join('..', 'data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
if not os.path.exists(raw_csv):
    raw_csv = os.path.join('data', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

df = pd.read_csv(raw_csv)
print('Dataset Shape:', df.shape)

## Phase 3: Exploratory Data Analysis (EDA) & Visualizations

In [ ]:
# Fix TotalCharges data type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Target Distribution
print('Churn Distribution:')
print(df['Churn'].value_counts(normalize=True) * 100)

# Numerical Summary by Churn
print('\nNumerical Feature Means by Churn:')
print(df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].mean())

# Plot Numerical Distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
for i, col in enumerate(num_cols):
    sns.kdeplot(data=df, x=col, hue='Churn', common_norm=False, fill=True, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
plt.tight_layout()
plt.show()

# Plot Categorical Churn Rates
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=df, x='Contract', y=(df['Churn']=='Yes').astype(int), hue='Contract', ax=axes[0], palette='Reds_r', errorbar=None, legend=False)
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold')
axes[0].set_ylim(0, 0.5)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height()*100:.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')

sns.barplot(data=df, x='InternetService', y=(df['Churn']=='Yes').astype(int), hue='InternetService', ax=axes[1], palette='Oranges_r', errorbar=None, legend=False)
axes[1].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[1].set_ylim(0, 0.5)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height()*100:.1f}%', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Phase 4: Data Preprocessing & ColumnTransformer Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 
    'PhoneService', 'MultipleLines', 'InternetService', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
    'TechSupport', 'StreamingTV', 'StreamingMovies', 
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

print('Preprocessor constructed successfully.')

## Phase 5: Stratified Train / Validation / Test Splitting

In [ ]:
from sklearn.model_selection import train_test_split

X = df[num_cols + cat_cols]
y = df['Churn'].map({'Yes': 1, 'No': 0})

# 70/15/15 Stratified Split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15/0.85, stratify=y_train_val, random_state=42)

print('Train shape :', X_train.shape, '| Churn rate:', round(y_train.mean()*100, 2), '%')
print('Val shape   :', X_val.shape, '| Churn rate:', round(y_val.mean()*100, 2), '%')
print('Test shape  :', X_test.shape, '| Churn rate:', round(y_test.mean()*100, 2), '%')

## Phase 6 & 7: Model Training, Threshold Tuning & Test Evaluation Plots

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_recall_curve, auc, confusion_matrix

# Create & Fit Pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
model_pipeline.fit(X_train, y_train)

# Threshold Tuning on Validation Split ONLY
val_probas = model_pipeline.predict_proba(X_val)[:, 1]
best_t, best_val_f1 = 0.50, 0.0
for t in np.linspace(0.10, 0.90, 81):
    score = f1_score(y_val, (val_probas >= t).astype(int))
    if score > best_val_f1:
        best_val_f1 = score
        best_t = t

print('Optimal Threshold Selected on Validation:', round(best_t, 2))

# Final Evaluation on Test Set
test_probas = model_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probas >= best_t).astype(int)

test_prec_arr, test_rec_arr, _ = precision_recall_curve(y_test, test_probas)
test_pr_auc = auc(test_rec_arr, test_prec_arr)

print('\nTest Classification Report:')
print(classification_report(y_test, test_preds, target_names=['Retained', 'Churned']))
print(f'Test PR-AUC: {test_pr_auc:.4f}')

# Plot Confusion Matrix Heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Retained (0)', 'Churned (1)'], yticklabels=['Retained (0)', 'Churned (1)'])
axes[0].set_title(f'Confusion Matrix (Test Set, Threshold = {best_t:.2f})', fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Plot Precision-Recall Curve
axes[1].plot(test_rec_arr, test_prec_arr, color='#2980b9', lw=2, label=f'Logistic Regression (PR-AUC = {test_pr_auc:.3f})')
axes[1].axhline(y=y_test.mean(), color='red', linestyle='--', label=f'Baseline (No Skill = {y_test.mean():.3f})')
axes[1].set_title('Precision-Recall Curve (Test Set)', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')
plt.tight_layout()
plt.show()

## Phase 8 & 9: Subgroup Error Analysis

In [ ]:
# Subgroup Error Analysis by Contract Type
test_df_analysis = X_test.copy()
test_df_analysis['Actual_Churn'] = y_test
test_df_analysis['Predicted_Churn'] = test_preds

print('--- ERROR RATES BY CONTRACT TYPE ---')
for contract, group in test_df_analysis.groupby('Contract'):
    total = len(group)
    fn = ((group['Actual_Churn'] == 1) & (group['Predicted_Churn'] == 0)).sum()
    fp = ((group['Actual_Churn'] == 0) & (group['Predicted_Churn'] == 1)).sum()
    actual_churned = (group['Actual_Churn'] == 1).sum()
    actual_retained = (group['Actual_Churn'] == 0).sum()
    fpr_str = f'{fp/actual_retained:.1%}' if actual_retained > 0 else '0%'
    fnr_str = f'{fn/actual_churned:.1%}' if actual_churned > 0 else '0%'
    print(f'Contract: {contract:15s} | Total: {total:3d} | FPR: {fpr_str:6s} | FNR: {fnr_str:6s}')